# Operating System Random Number Generation & Statistical Analysis
**Course**: Cryptography Laboratory (22CPP307)  
**Objective**: Evaluate operating-system random-number generation mechanisms across Linux and Windows, apply statistical test batteries (NIST SP 800-22, Chi-Square, Serial Correlation), and investigate their suitability for cryptographic applications.

---
## Team Members (Group 13)
- **Ayush Sthapak** (2024UCP1353)
- **Rishika Vijay** (2024UCP1398)

---
## Workflow & Methodology
1. **Dataset Generation**: 1 MiB (1,048,576 bytes) generated using:
   - Linux `/dev/urandom`
   - OpenSSL `rand`
   - Windows CNG / .NET `RandomNumberGenerator`
2. **Basic Statistical Analysis**: Byte frequencies, bit distributions, and repeated pattern analysis.
3. **Advanced Statistical Randomness**: Serial correlation, Pearson's Chi-Square uniformity test, NIST Monobit test, and NIST Runs test.
4. **Architectural & Security Research**: Linux CSPRNG vs Windows CNG, and why statistical tests do not guarantee cryptographic security.


In [ ]:
import os
import math
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

# Configure matplotlib formatting
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10


## 1. Dataset Loading & Verification
We load the three 1 MiB binary datasets generated via OS-level CSPRNGs:
- `linux_urandom.bin`: Direct kernel entropy output from Linux `/dev/urandom`.
- `openssl_rand.bin`: User-space cryptographic generator from OpenSSL.
- `windows_cng.bin`: Windows Cryptography Next Generation (`BCryptGenRandom` / .NET `RandomNumberGenerator`).


In [ ]:
data_dir = 'data'
datasets = {
    'Linux (/dev/urandom)': os.path.join(data_dir, 'linux_urandom.bin'),
    'OpenSSL (rand)': os.path.join(data_dir, 'openssl_rand.bin'),
    'Windows (CNG / .NET)': os.path.join(data_dir, 'windows_cng.bin')
}

raw_data = {}
byte_arrays = {}

print(f"{'Dataset':<25} | {'File Size (Bytes)':<18} | {'Total Bits':<15} | {'Integrity'}")
print("-" * 72)
for name, path in datasets.items():
    with open(path, 'rb') as f:
        data = f.read()
    raw_data[name] = data
    byte_arrays[name] = np.frombuffer(data, dtype=np.uint8)
    size = len(data)
    bits = size * 8
    is_valid = (size == 1048576)
    print(f"{name:<25} | {size:<18,} | {bits:<15,} | {'VALID (Exact 1 MiB)' if is_valid else 'INVALID'}")


## 2. Basic Statistical Analysis
### 2.1 Byte Frequency Distribution
In an ideal random byte stream, each of the 256 possible byte values (0 to 255) should occur with equal probability $p = 1/256$.
For a 1 MiB dataset ($N = 1,048,576$ bytes):
$$\text{Expected Frequency } E = \frac{1,048,576}{256} = 4096.0\text{ occurrences per byte}$$


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

for ax, (name, arr) in zip(axes, byte_arrays.items()):
    counts = np.bincount(arr, minlength=256)
    ax.bar(range(256), counts, color='#1f77b4', width=1.0, alpha=0.85)
    ax.axhline(y=4096, color='red', linestyle='--', linewidth=1.5, label='Expected E=4096')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Byte Value (0 - 255)')
    ax.set_ylabel('Observed Count')
    ax.legend(loc='upper right')
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle('Byte Frequency Distribution (Expected = 4096 per byte)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### 2.2 Bit Distribution (0s vs 1s) and Repeated Patterns
A random binary stream requires an exact 50% balance of 0 and 1 bits. Furthermore, the stream must not exhibit abnormal repeated sequences or periodic patterns.


In [ ]:
print(f"{'Dataset':<25} | {'Zeros (0s)':<12} | {'Ones (1s)':<12} | {'Ratio (1s)':<10} | {'Max Identical Repeat'}")
print("-" * 80)

for name, arr in byte_arrays.items():
    bits = np.unpackbits(arr)
    ones = int(np.sum(bits))
    zeros = len(bits) - ones
    ratio = (ones / len(bits)) * 100
    
    # Calculate longest identical consecutive run of bytes
    diffs = np.where(arr[:-1] != arr[1:])[0]
    run_lengths = np.diff(np.concatenate(([-1], diffs, [len(arr)-1])))
    max_repeat = int(np.max(run_lengths))
    
    print(f"{name:<25} | {zeros:<12,} | {ones:<12,} | {ratio:<9.3f}% | {max_repeat} bytes")


## 3. Advanced Statistical Randomness Tests
We evaluate each dataset against four rigorous statistical randomness criteria:

1. **Serial Correlation Coefficient ($r$)**:
   Measures linear dependence between adjacent bytes ($x_i$ and $x_{i+1}$). Ideal random data yields $r \approx 0$.
   $$r = \frac{\sum_{i=0}^{N-2} (x_i - \bar{x})(x_{i+1} - \bar{x})}{\sum_{i=0}^{N-1} (x_i - \bar{x})^2}$$

2. **Pearson's $\chi^2$ Goodness-of-Fit Test**:
   Tests whether byte distribution conforms to a discrete uniform distribution across $df = 255$ degrees of freedom.
   $$\chi^2 = \sum_{i=0}^{255} \frac{(O_i - E)^2}{E}$$
   A $P$-value $\ge 0.01$ indicates statistical uniformity.

3. **NIST SP 800-22 Monobit (Frequency) Test**:
   Tests whether the total proportion of $0$s and $1$s in the entire $8,388,608$-bit sequence matches a truly random source:
   $$S_{obs} = \frac{|S_n|}{\sqrt{n}}, \quad P\text{-value} = \text{erfc}\left(\frac{S_{obs}}{\sqrt{2}}\right)$$

4. **NIST SP 800-22 Runs Test**:
   Examines whether the number of runs of consecutive identical bits conforms to random oscillation expectations:
   $$V_n = 1 + \sum_{k=0}^{n-2} (b_k \neq b_{k+1}), \quad P\text{-value} = \text{erfc}\left(\frac{|V_n - 2n\pi(1-\pi)|}{2\sqrt{2n}\pi(1-\pi)}\right)$$


In [ ]:
import sys
sys.path.append('.')
from statistical_tests import (
    compute_serial_correlation,
    compute_chi_square,
    compute_monobit_test,
    compute_runs_test
)

results = []
for name, data in raw_data.items():
    sc = compute_serial_correlation(data)
    chi2_val, p_chi2, pass_chi2 = compute_chi_square(data)
    ones, zeros, s_obs, p_monobit, pass_monobit = compute_monobit_test(data)
    runs_obs, runs_exp, p_runs, pass_runs = compute_runs_test(data)
    
    results.append({
        'name': name,
        'sc': sc,
        'chi2': chi2_val,
        'p_chi2': p_chi2,
        's_obs': s_obs,
        'p_monobit': p_monobit,
        'p_runs': p_runs,
        'passed': pass_chi2 and pass_monobit and pass_runs and (abs(sc) < 0.005)
    })

print(f"{'Dataset':<22} | {'Serial (r)':<11} | {'Chi2 (p-val)':<14} | {'Monobit (p)':<12} | {'Runs (p-val)':<13} | {'Verdict'}")
print("=" * 88)
for r in results:
    chi2_str = f"{r['chi2']:.1f} ({r['p_chi2']:.3f})"
    mono_str = f"{r['p_monobit']:.4f}"
    runs_str = f"{r['p_runs']:.4f}"
    verdict = 'PASS (Random)' if r['passed'] else 'FAIL'
    print(f"{r['name']:<22} | {r['sc']:<11.6f} | {chi2_str:<14} | {mono_str:<12} | {runs_str:<13} | {verdict}")


## 4. Operating System RNG Comparison & Security Analysis

### 4.1 Linux CSPRNG Architecture
- **Engine**: ChaCha20 stream cipher operating in kernel space (`drivers/char/random.c`).
- **Interfaces**:
  - `/dev/urandom`: Standard, high-throughput non-blocking device interface.
  - `getrandom(2)`: System call that guarantees the kernel entropy pool is fully initialized prior to emitting bytes.
- **Entropy Sources**: Hardware interrupt timings, input devices, CPU jitter, and `RDRAND`/`RDSEED`.

### 4.2 Windows Cryptography Next Generation (CNG)
- **Engine**: NIST SP 800-90A **AES-256 CTR_DRBG** implemented in `ksecdd.sys`.
- **Interfaces**:
  - `BCryptGenRandom` using `BCRYPT_USE_SYSTEM_PREFERRED_RNG`.
  - `.NET` `System.Security.Cryptography.RandomNumberGenerator`.
- **Entropy Sources**: TPM 2.0, high-resolution performance counters (`QueryPerformanceCounter`), `RDTSC` jitter, and hardware interrupts.

---

### 4.3 Why Passing Statistical Tests Does NOT Guarantee Cryptographic Security
A critical principle of cryptographic engineering is that **statistical randomness is necessary, but fundamentally insufficient for security**.

1. **Black-Box Limitation**:
   Statistical tests treat data as a black-box byte string. They verify distributional properties (equal frequency, lack of simple periodicity) but cannot assess computational unpredictability or key secrecy.

2. **Counterexample 1 — Mersenne Twister (MT19937)**:
   MT19937 achieves near-perfect scores on Chi-Square, Monobit, and Runs tests. However, because its state transitions are linear over $\text{GF}(2)$, an adversary observing $624$ consecutive 32-bit outputs can fully reconstruct the $19,937$-bit internal state and predict every future output.

3. **Counterexample 2 — Fixed-Key Block Cipher in Counter Mode**:
   A deterministic generator running $\text{AES}_{K=0}(0), \text{AES}_{K=0}(1), \dots$ passes all NIST statistical tests with flying colors because AES is a strong pseudorandom permutation. Yet it provides zero security against an attacker who knows the fixed key $K=0$.

4. **Counterexample 3 — Insufficient Seed Entropy**:
   A CSPRNG seeded with a low-entropy seed (such as a 16-bit integer or timestamp) produces statistically flawless output that can nonetheless be brute-forced in under a tenth of a second.

---

## 5. Summary & Conclusion
All three operating-system random sources—Linux `/dev/urandom`, OpenSSL `rand`, and Windows CNG—passed all statistical randomness batteries (Serial Correlation $\approx 0$, Pearson $\chi^2$ $P \ge 0.01$, NIST Monobit $P \ge 0.01$, and NIST Runs $P \ge 0.01$).

Both modern operating systems incorporate forward-secure, prediction-resistant CSPRNGs backed by kernel-level entropy collection. Developers must always rely on these OS cryptographic primitives (`getrandom`, `BCryptGenRandom`, `secrets`, or `RandomNumberGenerator`) rather than non-cryptographic PRNGs (`random`, `rand()`, `Math.random()`).
